# Grid Capex-to-Alpha — U.S. Electrical Equipment
### Research-grade public-data pilot · October 19, 2026

**Question:** Does a fixed-scale gap between U.S. power-system output and electrical-equipment manufacturing output forecast equipment production six months ahead?

**Ex ante falsification standard:** the factor must beat both a momentum-only benchmark and a historical-mean benchmark, survive time-shift placebos, and be reconstructed with point-in-time vintages before any alpha claim.

**Preliminary verdict:** A structural demand case exists, but the gap factor does not beat the mean benchmark (full RMSE 4.124 pp vs mean 3.524 pp) on 19 overlapping held-out forecast origins. **No security-level returns are analysed.**

Refer to README.md for source register, hypothesis definitions, model risk, limitations and future research.

## 1 · Set-up and provenance

The included monthly government index values are a 2022-2026 official-table transcription. They are **not** an ALFRED point-in-time panel and not a data-centre load time series. Both the static source data and all execution logic are included in this directory.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from grid_alpha import read_snapshot, make_panel, analyse, walk_forward, shifted_placebo, block_bootstrap
DATA = Path('fed_industrial_production_snapshot_2022_2026.csv')
raw = read_snapshot(DATA)
raw.tail()

## 2 · Fixed signal and lagged feature engineering

At formation month t only month t−1 macro features are permitted. `gap = tanh((power_yoy - equip_yoy) / 10)`. Target = next-six-month percentage change in equipment IP. The parameter 10 pp is fixed before inspection; no all-sample mean/scaling enters construction.

In [ ]:
panel = make_panel(raw)
assert len(panel) == 37
panel[['gap', 'load_yoy_pit', 'equip_yoy_pit', 'y']].head()

## 3 · Expanding OOS predictions and baselines

The training set for an origin t is restricted to labels with formation dates no later than t−7 months (six-month target maturity + one publication month). A train-only standardized Ridge with equipment growth/momentum is compared with the same model + gap and a simple expanding historical mean. These are **economic forecasts**, not position returns.

In [ ]:
results, panel, oos = analyse(DATA)
import json
print(json.dumps(results, indent=2, ensure_ascii=False))
oos.tail(10)

## 4 · Placebo, dependence and bias diagnostics

Circularly shift the observed gap series by 7+ periods and refit the *entire* walk-forward sequence. This checks some alignment fragility but is **not** an exact time-series randomization test. Additionally resample six-month forecast errors in blocks; the tiny and overlapping OOS sample is not adequate for credible confidence bounds.

In [ ]:
print('Placebo:', results['circular_shift_placebo'])
print('Block-bootstrap illustrative interval:', results['block_bootstrap_mse_improvement_95pct_exploratory'])
print('Rank correlation (nominal only):', results['full_sample_spearman'], results['naive_nominal_spearman_p'])

## 5 · Visual audit

Latest-revised production indices can co-move in ways unrelated to AI. Display outcomes and forecast errors; avoid presenting a cumulative P&L for data that do not include investable returns.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8))
raw[['load', 'equip']].plot(ax=axes[0], title='Federal Reserve industrial-production indices (latest/mixed vintages)')
oos[['actual', 'mean', 'baseline', 'with_gap']].plot(ax=axes[1], title='Next-six-month equipment output growth, percentage points')
plt.tight_layout()
plt.show()

## 6 · Equity alpha stage gate — NOT PASSED

Necessary data not present: contemporaneous security universes; adjusted total returns and dividend/delisting information; release-known stock-to-industry exposure weights; regional interconnection and load data; financing/trading costs. **Do not convert this macro prediction exercise into a claim of stock-market alpha.**

A future version should test 10-K-observable exposures against next-quarter earnings revisions, sector-neutral rank IC and fully adjusted portfolio returns under historical vintage constraints. See README.md §§7–9.